<a href="https://colab.research.google.com/github/tk-learning-artifacts/ai-accelerator-2026-notebooks/blob/main/student/x-diffusion-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beyond the course · Diffusion 1 — Make digits out of noise

Image generators that make pictures from nothing work by a surprising trick.
Take a real picture and add a little random noise, again and again, until
only noise is left. Then train a network to **undo one step** of that. Start
from pure noise, undo step after step, and a brand-new picture appears. These
are **diffusion models**.

In this notebook you build one from scratch, small enough to train in under a
minute or two on a free Colab CPU. It learns from the 8×8 handwritten digits you met
on Wednesday, and by the end it draws new digits, including the digit you ask
for.

This is an extension pathway: you work through it alone or with a partner,
at your own pace. You need Wednesday's material (a network, a loss, the
training loop). You can start it any time after Wednesday. Allow about two
hours.

How it works, as in class: 🔮 predict before you run, ✏️ replace the `...`,
✅ run the check under it. A check that fails tells you what to look at. Each
section ends with a **You're done when** line. No GPU needed.

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression


def is_done(*values):
    """True once none of the values is still `...`."""
    return all(v is not ... for v in values)


def check(ok, hint):
    if not ok:
        raise AssertionError("Not yet. " + hint)
    print("✅ Correct!")


# The instructors' checker sets NB_CHECK to run this notebook quickly; ignore it.
QUICK = os.environ.get("NB_CHECK") == "1"
torch.manual_seed(0)
np.random.seed(0)

digits = load_digits()
# Pixels go from 0 to 16; we rescale them to run from -1 (white) to 1 (black).
real = torch.tensor(digits.data / 8 - 1, dtype=torch.float32)   # 1,797 images, 64 numbers each
labels = torch.tensor(digits.target)


def show(images, rows=1, titles=None, size=1.2):
    """Draw images (each 64 numbers) in a grid."""
    images = images.detach().clamp(-1, 1).reshape(-1, 8, 8)
    cols = int(np.ceil(len(images) / rows))
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for k, ax in enumerate(axes.flat):
        ax.axis("off")
        if k < len(images):
            ax.imshow(images[k], cmap="gray_r", vmin=-1, vmax=1)
            if titles is not None:
                ax.set_title(titles[k], fontsize=8)
    plt.show()


print(real.shape, "from", real.min().item(), "to", real.max().item())
show(real[:16], titles=[str(int(d)) for d in labels[:16]])

## 1. Destroying a picture, step by step

The **forward process** adds a little noise at each of `T = 200` steps. The
amount added at step t is `betas[t]`: tiny at first, bigger later. One step
shrinks the picture a little and adds a little random noise:

new = √(1 − beta) × old + √beta × noise

where `noise` is a fresh random number for every pixel (from
`torch.randn`, average 0, spread 1). Here is that rule applied 200 times to
one digit:

In [ ]:
T = 200
betas = torch.linspace(1e-4, 0.05, T)

x = real[0].clone()
snapshots, steps_shown = [x], [0]
for t in range(T):
    x = (1 - betas[t]).sqrt() * x + betas[t].sqrt() * torch.randn(64)
    if t + 1 in (10, 25, 50, 100, 200):
        snapshots.append(x)
        steps_shown.append(t + 1)
show(torch.stack(snapshots), titles=[f"step {s}" for s in steps_shown])

Doing 100 small steps to get step 100 is slow. Because each step only
shrinks and adds noise, the steps combine into one. Multiply all the
`(1 − beta)`s up to step t; call the product `keep[t]`. Then

noisy at step t = √keep[t] × clean + √(1 − keep[t]) × noise

with **one** fresh noise image. `keep[t]` starts near 1 (almost all picture)
and ends near 0 (almost all noise). `torch.cumprod` multiplies as it goes,
like a running total but with ×:

In [ ]:
keep = torch.cumprod(1 - betas, dim=0)
for t in (0, 25, 50, 100, 150, 199):
    print(f"step {t:3d}: keep {keep[t]:.3f}  ->  {keep[t].sqrt():.2f} × picture + {(1 - keep[t]).sqrt():.2f} × noise")

🔮 **Predict.** Before running the next cells: at step 50 the picture's
share is 0.85 and the noise's 0.53; at step 100 they have swapped. Will you
still be able to read the digit at step 50? At step 100?

In [ ]:
my_prediction = ""  # @param {type:"string"}

✏️ **Exercise 1a.** Write `add_noise(clean, t, noise)` using the one-step
formula above. `t` is a tensor of step numbers, one for each image, so
`keep[t]` has one number per image; `[:, None]` turns it into a column so it
multiplies every pixel of its own image.

In [ ]:
def add_noise(clean, t, noise):
    k = keep[t][:, None]
    return ...

In [ ]:
c, e = torch.ones(3, 64), torch.zeros(3, 64)
t3 = torch.tensor([0, 100, 199])
out = add_noise(c, t3, e)
check(is_done(out), "add_noise still returns `...`.")
check(torch.allclose(out[:, 0], keep[t3].sqrt()), "With zero noise, an all-ones picture should become √keep[t] at every pixel. Is the picture multiplied by keep[t].sqrt()?")
check(torch.allclose(add_noise(e, t3, c)[:, 0], (1 - keep[t3]).sqrt()), "With a blank picture, the result should be √(1 − keep[t]) × noise.")

In [ ]:
t_demo = torch.tensor([0, 10, 25, 50, 75, 100, 150, 199])
show(add_noise(real[:1].repeat(8, 1), t_demo, torch.randn(8, 64)), titles=[f"step {int(t)}" for t in t_demo])

**You're done when** your `add_noise` check passes and you can say from the
pictures roughly where the digit stops being readable.

## 2. The denoiser

The network's job: given a noisy picture **and the step number**, guess the
noise that was added. If it can guess the noise, it can subtract it.

It is Wednesday's network with more inputs. The step number matters (at step
5 the noise is faint; at step 190 it is almost everything), so the network
gets it too, written as 32 numbers instead of one: the sine and cosine of the
step at 16 different speeds, like the hands of a clock that turn at different
rates. Networks learn more easily from that than from a single number from
0 to 199.

In [ ]:
def step_features(t):
    """32 numbers describing each step number in t."""
    speeds = torch.exp(-torch.arange(16) * 0.5) * 50
    angle = t[:, None].float() / T * speeds[None, :]
    return torch.cat([angle.sin(), angle.cos()], dim=1)


print(step_features(torch.tensor([0, 100, 199])).shape)

🔮 **Predict.** The network takes the noisy picture (64 numbers) and the step
features (32 numbers) side by side. How many numbers go into its first
layer? How many come out of its last layer?

In [ ]:
my_prediction = ""  # @param {type:"string"}

✏️ **Exercise 2a.** In `forward`, join the noisy pictures `x` and the step
features of `t` side by side with `torch.cat(..., dim=1)`, then pass the
result through `self.net`.

In [ ]:
class Denoiser(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64 + 32, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 64),
        )

    def forward(self, x, t):
        both = ...
        return self.net(both)

`nn.SiLU` is a smoother cousin of ReLU, common in diffusion models.

In [ ]:
try:
    out = Denoiser()(torch.zeros(5, 64), torch.tensor([0, 1, 2, 3, 4]))
except TypeError:
    out = ...
check(is_done(out), "forward still has a `...`. Join x and step_features(t).")
check(tuple(out.shape) == (5, 64), f"5 pictures in should give 5 guesses of 64 numbers, not {tuple(out.shape)}.")

Everyone continues from here (code hidden; double-click to see it after).

In [ ]:
#@title Run this to continue { display-mode: "form" }
def add_noise(clean, t, noise):
    share = keep[t].unsqueeze(1)
    return share.sqrt() * clean + (1 - share).sqrt() * noise


class Denoiser(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(96, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 64),
        )

    def forward(self, x, t):
        return self.net(torch.cat((x, step_features(t)), 1))

**You're done when** the shape check passes and you can explain why the
network needs the step number.

## 3. Train it to guess the noise

One training example is made like this: take real digits, pick a random step
for each, make fresh noise, and mix them with `add_noise`. The network sees
the noisy digit and the step, and the loss compares its guess with the noise
that was really added: Wednesday's mean squared error.

✏️ **Exercise 3a.** Finish `diffusion_loss(model, clean)`. The random steps and
the noise are made for you. Make the noisy pictures with your `add_noise`,
then return the mean squared error between `model(noisy, t)` and `noise`.

In [ ]:
def diffusion_loss(model, clean):
    t = torch.randint(T, (len(clean),))
    noise = torch.randn_like(clean)
    noisy = ...
    return ...

🔮 **Predict.** A lazy network that always guesses "no noise" (all zeros):
what loss will it get? (The noise has average 0 and spread 1.)

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
class GuessZero(nn.Module):
    def forward(self, x, t):
        return torch.zeros_like(x)


lazy = diffusion_loss(GuessZero(), real[:1000])
check(is_done(lazy), "diffusion_loss still returns `...`.")
check(torch.is_tensor(lazy) and lazy.dim() == 0, "Return a single number: the .mean() of the squared differences.")
check(0.9 < lazy.item() < 1.1, f"A model that guesses zeros should score about 1.0 (the noise's spread squared), not {lazy.item():.2f}. Compare the guess with `noise`, not with `clean`.")

Everyone continues from here.

In [ ]:
#@title Run this to continue { display-mode: "form" }
def diffusion_loss(model, clean):
    t = torch.randint(T, (len(clean),))
    noise = torch.randn_like(clean)
    guess = model(add_noise(clean, t, noise), t)
    return nn.functional.mse_loss(guess, noise)

Now the five lines from Wednesday, 6,000 times, on batches of 128 random
digits. This takes a minute or two on Colab's free CPU.

In [ ]:
STEPS = 50 if QUICK else 6000
model = Denoiser()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
losses, start = [], time.time()
for step in range(STEPS):
    batch = real[torch.randint(len(real), (128,))]
    optimizer.zero_grad()
    loss = diffusion_loss(model, batch)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step % 1000 == 0 or step == STEPS - 1:
        print(f"step {step:5d}  loss {loss.item():.3f}  {time.time() - start:.0f} s")

plt.figure(figsize=(6, 3))
plt.plot(np.convolve(losses, np.ones(50) / 50, mode="valid"))
plt.xlabel("step"); plt.ylabel("loss (average of 50)")
plt.show()

🤔 **Think.** The loss falls fast and then flattens well above 0. Why can't
even a perfect denoiser get the loss to 0? (Think about step 199, where the
picture is almost all noise.)

**You're done when** your loss is well below the lazy model's 1.0 and has
flattened out.

## 4. Run it backwards: make new digits

If the network guesses the noise, you can estimate the clean picture by
running `add_noise` backwards. This is the worked example for the exercise
after it:

clean guess = (noisy − √(1 − keep[t]) × noise guess) / √keep[t]

In [ ]:
t_one = torch.tensor([60] * 8)
clean = real[8:16]
noisy = add_noise(clean, t_one, torch.randn(8, 64))
with torch.no_grad():
    guess_noise = model(noisy, t_one)
k = keep[t_one][:, None]
clean_guess = (noisy - (1 - k).sqrt() * guess_noise) / k.sqrt()
show(torch.cat([clean, noisy, clean_guess]), rows=3)
print("rows: the real digits, noised to step 60, the network's guess of the clean digits")

The guess from a single jump is blurry. Generating works better in small
steps: from step t, remove a fraction of the guessed noise to get to step
t − 1, add back a little fresh noise, and repeat 200 times from pure noise.
One step of the reverse process is

previous = (x − beta / √(1 − keep[t]) × noise guess) / √(1 − beta) + √beta × fresh noise

with `beta = betas[t]`. At the last step (t = 0) no fresh noise is added.

✏️ **Exercise 4a.** Write `reverse_step(x, t, noise_guess, fresh)` for a
whole batch at the same step `t` (an ordinary whole number). `fresh` is the
fresh noise, passed in so the check can test your formula. Don't add it when
`t == 0`.

In [ ]:
def reverse_step(x, t, noise_guess, fresh):
    beta = betas[t]
    previous = ...
    return ...

In [ ]:
x_t, g, z = torch.full((2, 64), 0.5), torch.full((2, 64), 0.2), torch.ones(2, 64)
out = reverse_step(x_t, 120, g, z)
check(is_done(out), "reverse_step still returns `...`.")
expected = (0.5 - betas[120] / (1 - keep[120]).sqrt() * 0.2) / (1 - betas[120]).sqrt() + betas[120].sqrt()
check(torch.allclose(out, expected.expand(2, 64)), "At step 120 your result doesn't match the formula. Check the brackets, and that fresh noise is multiplied by √beta.")
expected0 = (0.5 - betas[0] / (1 - keep[0]).sqrt() * 0.2) / (1 - betas[0]).sqrt()
check(torch.allclose(reverse_step(x_t, 0, g, z), expected0.expand(2, 64)), "At t = 0, add no fresh noise.")

Everyone continues from here.

In [ ]:
#@title Run this to continue { display-mode: "form" }
def reverse_step(x, t, noise_guess, fresh):
    b = betas[t]
    out = (x - b * noise_guess / (1 - keep[t]).sqrt()) / (1 - b).sqrt()
    return out + b.sqrt() * fresh if t > 0 else out


@torch.no_grad()
def generate(model, n, snapshots_at=()):
    x = torch.randn(n, 64)                         # step 199: pure noise
    snapshots = []
    for t in reversed(range(T)):
        x = reverse_step(x, t, model(x, torch.full((n,), t)), torch.randn(n, 64))
        if t in snapshots_at:
            snapshots.append(x.clone())
    return x, snapshots

🔮 **Predict.** Watch one picture come out of the noise. At which step do you
expect to first recognise the digit: near 150, near 100, or only near the
end?

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
at = (199, 150, 100, 60, 30, 10, 0)
final, snaps = generate(model, 1, snapshots_at=at)
show(torch.cat(snaps), titles=[f"step {t}" for t in at])

new_digits, _ = generate(model, 64)
show(new_digits, rows=4)

These digits are new: the model never saw them. Are they any good? Ask a
judge that has never seen the model: a simple classifier trained on the real
digits. If a sample looks like a clear digit, the judge is confident about
it. If it looks like a scribble, the judge hesitates.

In [ ]:
# The judge learns from the first 1,000 real digits only, so we can test it fairly on others.
judge = LogisticRegression(max_iter=3000).fit(digits.data[:1000], digits.target[:1000])


def judge_samples(samples):
    pixels = ((samples.clamp(-1, 1) + 1) * 8).numpy()   # back to 0–16
    probs = judge.predict_proba(pixels)
    return probs.argmax(1), probs.max(1)


guess, confidence = judge_samples(new_digits)
real_guess, real_conf = judge_samples(real[1000:1064])
print(f"judge's average confidence:  real digits {real_conf.mean():.2f}   your model's {confidence.mean():.2f}")
print("how many of each digit the model drew:", np.bincount(guess, minlength=10))

**You're done when** you have a grid of 64 generated digits and the judge's
confidence for them, and you can say which digits your model draws most and
least often.

## 5. Ask for a digit

Your model draws a random digit. To ask for a 7, give the denoiser the label
as one more input: ten numbers, all 0 except a 1 in the label's place (a
**one-hot** vector). An eleventh slot means "no label".

**Classifier-free guidance** makes the model stick to the label much more
closely. During training, the label is hidden (replaced by "no label") one
time in ten, so the same network learns two jobs: guess the noise for a 7,
and guess it for any digit. When generating, ask both and push away from the
"any digit" guess, towards the "7" guess:

guided = any-digit guess + w × (7 guess − any-digit guess)

`w` is the **guidance scale**. With w = 1 it is the plain "7" guess; above 1
it exaggerates whatever makes a 7 different from a digit in general.

The labelled model is defined and trained for you. It is the same as before,
with 11 more inputs. Training takes about a minute.

In [ ]:
class LabelledDenoiser(nn.Module):
    def __init__(self, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64 + 32 + 11, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, 64),
        )

    def forward(self, x, t, label):
        """label is 0–9, or 10 for 'no label'."""
        one_hot = nn.functional.one_hot(label, 11).float()
        return self.net(torch.cat([x, step_features(t), one_hot], dim=1))


NO_LABEL = 10
STEPS = 50 if QUICK else 8000
cmodel = LabelledDenoiser()
optimizer = torch.optim.Adam(cmodel.parameters(), lr=2e-3)
start = time.time()
for step in range(STEPS):
    pick = torch.randint(len(real), (128,))
    clean, label = real[pick], labels[pick].clone()
    label[torch.rand(128) < 0.1] = NO_LABEL            # hide the label one time in ten
    t = torch.randint(T, (128,))
    noise = torch.randn_like(clean)
    optimizer.zero_grad()
    loss = nn.functional.mse_loss(cmodel(add_noise(clean, t, noise), t, label), noise)
    loss.backward()
    optimizer.step()
    if step % 2000 == 0 or step == STEPS - 1:
        print(f"step {step:5d}  loss {loss.item():.3f}  {time.time() - start:.0f} s")

✏️ **Exercise 5a.** Write `guide(with_label, without_label, w)` using the
guidance formula above.

In [ ]:
def guide(with_label, without_label, w):
    return ...

In [ ]:
a, b = torch.tensor([1.0, 2.0]), torch.tensor([0.0, 1.0])
check(is_done(guide(a, b, 1.0)), "guide still returns `...`.")
check(torch.allclose(guide(a, b, 1.0), a), "With w = 1 the result should be exactly the labelled guess.")
check(torch.allclose(guide(a, b, 0.0), b), "With w = 0 the result should be the 'no label' guess.")
check(torch.allclose(guide(a, b, 3.0), torch.tensor([3.0, 4.0])), "With w = 3: without + 3 × (with − without) = [3, 4].")

Everyone continues from here.

In [ ]:
#@title Run this to continue { display-mode: "form" }
def guide(with_label, without_label, w):
    return w * with_label + (1 - w) * without_label


@torch.no_grad()
def generate_digit(cmodel, digit, n, w):
    x = torch.randn(n, 64)
    asked = torch.full((n,), digit)
    blank = torch.full((n,), NO_LABEL)
    for t in reversed(range(T)):
        steps = torch.full((n,), t)
        g = guide(cmodel(x, steps, asked), cmodel(x, steps, blank), w)
        x = reverse_step(x, t, g, torch.randn(n, 64))
    return x

🔮 **Predict.** You will ask for each digit 0–9 with w = 0, 1 and 4. What
will the w = 0 row look like? Which row will the judge find the most
"on-label"? Is there a cost to a large w?

In [ ]:
my_prediction = ""  # @param {type:"string"}

In [ ]:
for w in (0.0, 1.0, 4.0):
    batch = torch.cat([generate_digit(cmodel, d, 20, w) for d in range(10)])
    asked = np.repeat(np.arange(10), 20)
    guess, confidence = judge_samples(batch)
    print(f"w = {w}: the judge agrees with the label asked for {np.mean(guess == asked):.0%} of the time "
          f"(average confidence {confidence.mean():.2f})")
    show(batch[::20], titles=[str(d) for d in range(10)])

Ask for your own: change `my_digit` and `my_w` and run again.

In [ ]:
my_digit = 7  # @param {type:"integer"}
my_w = 2.0  # @param {type:"number"}
show(generate_digit(cmodel, my_digit, 16, my_w), rows=2)

**You're done when** you can make a row of any digit you choose, and explain
in a sentence what the guidance scale changes.

### Stretch: make it yours

- **Copying?** For each generated digit, find the closest real digit (the
  smallest sum of squared differences over the 1,797). Show them side by side.
  Is the model inventing or remembering?
- **Fewer steps.** Change `T` to 50 (and rebuild `betas` and `keep` with a
  larger last beta, about 0.2), retrain and generate. What gets worse?
- **A bigger picture.** Diffusion 2, the next notebook in this pathway, does
  the same on 28×28 images with a U-Net (Segmentation 1's kind of network)
  as the denoiser.
- **Why this matters.** Big image generators use this same loop with a
  far bigger U-Net or transformer, and a text description in place of your
  ten-way label. Look at where your `guide` line would sit in such a model.

In [ ]:
# Your stretch work here.